# Lesson 09 Lab — Benchmark Protocol and Timing Error

**Puzzle:** When warmup, synchronization, quantiles, and JIT change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates warmup, synchronization, quantiles, and JIT and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

Asynchronous launches make host wall time and device execution time different quantities. JIT compilation makes the first call different again. A usable report states the timer, warmup, repetitions, synchronization point, and distribution rather than printing one number.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["warmup, synchronization, quantiles, and JIT"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

Including the first JIT call in a steady-state average can dominate the result and invert the decision.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 9
LESSON_TITLE = 'Benchmark Protocol and Timing Error'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260822
}


## 5. Freeze the experiment

**Experiment:** Measure one first host call, then retain 40 CUDA-event samples and p20/p50/p80.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 349.088495131582,
  "secondary": 0.02012799959629774,
  "max_abs_error": 4.76837158203125e-07,
  "passed": true,
  "details": {
    "p20_ms": 0.019039999693632126,
    "p80_ms": 0.021279999613761903,
    "samples_ms": [
      0.03062400035560131,
      0.0225600004196167,
      0.024000000208616257,
      0.021536000072956085,
      0.020927999168634415,
      0.02006400004029274,
      0.020735999569296837,
      0.01974399946630001,
      0.022655999287962914,
      0.020640000700950623,
      0.02006400004029274,
      0.019039999693632126,
      0.01942400075495243,
      0.02175999991595745,
      0.021215999498963356,
      0.020767999812960625,
      0.02051199972629547,
      0.02195199951529503,
      0.026016000658273697,
      0.020255999639630318,
      0.019999999552965164,
      0.02067199908196926,
      0.01897599920630455,
      0.019071999937295914,
      0.018464000895619392,
      0.019967999309301376,
      0.020191999152302742,
      0.0192959997802

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| First host call | 349.0885 ms |
| Warm p50 | 0.0201 ms |
| Maximum absolute error | 4.768e-07 |
| Acceptance gate | true |


## 8. Explain without overclaiming

The first host-observed call was 349.09 ms; warm event timing was p20=0.0190, p50=0.0201, p80=0.0213 ms.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Publish cold and warm measurements as separate service concerns; never average them together.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 9,
  "title": "Benchmark Protocol and Timing Error",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260822
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 349.088495131582,
    "secondary": 0.02012799959629774,
    "max_abs_error": 4.76837158203125e-07,
    "passed": true,
    "details": {
      "p20_ms": 0.019039999693632126,
      "p80_ms": 0.021279999613761903,
      "samples_ms": [
        0.03062400035560131,
        0.0225600004196167,
        0.024000000208616257,
        0.021536000072956085,
        0.020927999168634415,
        0.02006400004029274,
        0.020735999569296837,
        0.01974399946630001,
        0.022655999287962914,
        0.020640000700950623,
        0.02006400004029274,
        0.01

## 10. Make the bounded decision

> Publish cold and warm measurements as separate service concerns; never average them together.

**Failure analysis:** Including the first JIT call in a steady-state average can dominate the result and invert the decision.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
